In [ ]:
BANNER = \
"""
# ----------------------------------------------------------------------
# PTS | Paulo Trigo Silva
# mLDm | MoP
# v01
# ----------------------------------------------------------------------
"""

In [ ]:
# ------------------------------------------------------------
# connect DuckDB to the integrated PostgreSQL data source
# ------------------------------------------------------------

import duckdb

# create an in-memory DuckDB connection
db = duckdb.connect()

# install and load the DuckDB PostgreSQL extension
# the INSTALL statement is required only the first time the extension is used
db.sql("""
INSTALL postgres;
LOAD postgres;
""")

# attach the PostgreSQL database to DuckDB
# the database will be accessible through the name "pg"
db.sql("""
ATTACH
'host=localhost port=5432 dbname=mldm user=mldm password=mldm'
AS pg
(TYPE postgres);
""")

print("PTS | DuckDB connected to PostgreSQL")

In [ ]:
# ------------------------------------------------------------
# create a flat view of the integrated data source
# ------------------------------------------------------------

# combine penguin and observation data for analytical exploration
# preserve the original temporal attribute from the source
# derive year as an additional attribute for later analytical decisions
db.sql("""
CREATE OR REPLACE TEMP VIEW penguins_source AS
SELECT
    o.id_penguin,
    o.date_time,
    EXTRACT( YEAR FROM o.date_time )::VARCHAR AS year,
    o.name_study AS study,
    o.sample_number,
    p.name_species AS species,
    o.name_island AS island,
    p.sex,
    o.bill_length_mm,
    o.bill_depth_mm,
    o.flipper_length_mm,
    o.body_mass_g,
    o.latitude,
    o.longitude,
    o.name_batch,
    b.name_lab AS lab
FROM pg.public.observation AS o JOIN pg.public.penguin AS p
     ON p.id = o.id_penguin
     LEFT JOIN pg.public.batch AS b
     ON b.name = o.name_batch;
""")

# check the number of observations represented in the view
db.sql("""
SELECT COUNT(*) AS n_observations
FROM penguins_source;
""").show()

In [ ]:
# ------------------------------------------------------------
# inspect the flat integrated source structure
# ------------------------------------------------------------

# inspect attribute names and data types
schema = db.sql("""
DESCRIBE penguins_source;
""")

schema.project("""
column_name,
column_type
""").show()

# inspect a few observations
db.sql("""
SELECT *
FROM penguins_source
LIMIT 10;
""").show()

In [ ]:
# ------------------------------------------------------------
# guiding questions
# ------------------------------------------------------------

# IMPORTANT:
# these questions are not answered immediately
# they guide the construction of the following analysis cells
# each following cell should collect evidence relevant to one or more of these questions
# conclusions should be drawn only after the corresponding evidence has been inspected

# GQ1. how is the target species distributed?
# GQ2. is species associated with the island context?
# GQ3. do morphological measurements differ across species?
# GQ4. how does species vary across collection years, and is year analytically relevant?

In [ ]:
# ------------------------------------------------------------
# inspect the target distribution
# ------------------------------------------------------------

# count observations for each species
db.sql("""
SELECT
    species,

    -- TODO: count the number of observations for each species
    ... AS n_observations

FROM penguins_source

-- TODO: group observations by species
GROUP BY ...

ORDER BY n_observations DESC;
""").show()

# expected result:
# three rows, one for each species
# Adelie should be the most frequent class
# Chinstrap should be the least frequent class

In [ ]:
# ------------------------------------------------------------
# TODO
# interpret evidence for GQ1 — target distribution
# ------------------------------------------------------------

# task:
# - use the target counts obtained above
# - answer the evidence question and conclude on GQ1

# E1. which species are most and least represented, and what is the approximate largest-to-smallest ratio?
# A1:

# conclusion for GQ1
# - how is the target species distributed?
# A:

In [ ]:
# ------------------------------------------------------------
# construct a multidimensional analytical view
# ------------------------------------------------------------

# create the multidimensional aggregation
db.sql("""
CREATE OR REPLACE TEMP VIEW penguins_cube AS
SELECT
    species,
    island,
    sex,
    COUNT(*) AS n_observations,
    ROUND( AVG( bill_length_mm ), 2 ) AS avg_bill_length_mm,
    ROUND( AVG( bill_depth_mm ), 2 ) AS avg_bill_depth_mm,
    ROUND( AVG( flipper_length_mm ), 2 ) AS avg_flipper_length_mm,
    ROUND( AVG( body_mass_g ), 2 ) AS avg_body_mass_g,

    -- TODO: identify the aggregation level generated for
    --       species, island and sex using GROUPING_ID
    RIGHT( ...::BIT::VARCHAR, 3 ) AS grouping_bits

FROM penguins_source

-- TODO: generate all aggregation levels for species, island and sex
GROUP BY ...;
""")

# inspect all aggregation levels generated by the cube
db.sql("""
SELECT *
FROM penguins_cube
ORDER BY grouping_bits, species, island, sex;
""").show()

# count the cells generated at each aggregation level
db.sql("""
SELECT
    grouping_bits,
    COUNT(*) AS n_cells
FROM penguins_cube
GROUP BY grouping_bits
ORDER BY grouping_bits;
""").show()

# expected result:
# 8 aggregation levels, represented by grouping_bits from 000 to 111
# the detailed level (000) contains the largest number of cells
# the grand-total level (111) contains a single cell

In [ ]:
# ------------------------------------------------------------
# interpret the cube structure
# ------------------------------------------------------------

# the CUBE contains all eight aggregation levels
# induced by three dimensions (species, island, sex)

# the number of cells at each level reflects:
# - the combinations actually present in the data
# - not all theoretically possible combinations

# note that NULL may have two different meanings in a CUBE:
# - a missing value already present in the source data
# - a dimension that has been aggregated at that level
#
# grouping_bits allows these two situations to be distinguished

# in this dataset:
# - species and island have three observed non-null values each
# - sex has two observed non-null values plus missing values
# - therefore, NULL for sex with its grouping bit = 0 represents missing data
# - NULL with the corresponding grouping bit = 1 represents aggregation

#
# questions to investigate:
# 1. why are fewer cells observed than theoretically possible at some aggregation levels?
# 2. which species × island combinations are absent from the data?

In [ ]:
# ------------------------------------------------------------
# construct all species × island combinations
# ------------------------------------------------------------

# generate the Cartesian product of the observed species and island values
db.sql("""
CREATE OR REPLACE TEMP VIEW possible_species_island AS
SELECT
    s.species,
    i.island
FROM
    (SELECT DISTINCT species FROM penguins_source) AS s

    -- TODO: combine every observed species with every observed island
    ...

    (SELECT DISTINCT island FROM penguins_source) AS i;
""")

db.sql("""
SELECT *
FROM possible_species_island
ORDER BY island, species;
""").show()

# expected result:
# 9 rows corresponding to all combinations
# of 3 observed species × 3 observed islands

In [ ]:
# ------------------------------------------------------------
# identify absent species × island combinations
# ------------------------------------------------------------

# compare all species × island combinations with those observed in the cube
# recall that grouping_bits = '001' selects the level where sex is aggregated

db.sql("""
SELECT
    p.species AS species_absent,
    p.island AS island_absent
FROM possible_species_island AS p

     -- TODO: keep all possible combinations and match those observed in the cube
     ... JOIN
     penguins_cube AS o

     -- TODO: match species and island at the species × island aggregation level
     ON ...

-- TODO: retain only combinations with no corresponding observed cube cell
WHERE ...

ORDER BY p.species, p.island;
""").show()

# expected result:
# 4 absent species × island combinations
# Adelie should not appear among them

In [ ]:
# ------------------------------------------------------------
# inspect the observed species × island combinations
# ------------------------------------------------------------

# pivot the species × island level, where sex is aggregated
db.sql("""
PIVOT (
    SELECT
        species,
        island,
        n_observations
    FROM penguins_cube
    WHERE grouping_bits = '001'
)

-- TODO: make species values become columns
ON ...

-- TODO: aggregate the number of observations in each cell
USING ...

-- TODO: keep islands as rows
GROUP BY ...

ORDER BY island;
""").show()

#
# in DuckDB PIVOT:
# - ON defines which attribute values become columns
# - GROUP BY defines which attribute values remain as rows
# - USING defines the measure aggregated inside each row × column cell

# expected result:
# 3 rows, one for each island
# some species × island cells should be NULL,
# corresponding to combinations not observed in the data

In [ ]:
# ------------------------------------------------------------
# TODO
# interpret evidence for GQ2 — island context
# ------------------------------------------------------------

# task:
# - use the absent combinations and pivot results
# - answer the evidence question and conclude on GQ2

# E1. which species × island combinations are absent?
# A1:

# conclusion for GQ2
# - what does this pattern indicate about the association between island and species?
# A:

In [ ]:
# ------------------------------------------------------------
# inspect morphological measurements across species
# ------------------------------------------------------------

# grouping_bits = '011' keeps species and aggregates island and sex
# AVG ignores NULL values, so each measurement is averaged over its available values

db.sql("""
SELECT
    species,
    avg_bill_length_mm,
    avg_bill_depth_mm,
    avg_flipper_length_mm,
    avg_body_mass_g
FROM penguins_cube
WHERE grouping_bits = '011'
ORDER BY species;
""").show()

In [ ]:
# ------------------------------------------------------------
# TODO
# interpret evidence for GQ3 — morphological measurements
# ------------------------------------------------------------

# task:
# - use the average measurements obtained above
# - identify the main species-level differences and conclude on GQ3

# E1. for each measurement, which species has an average that appears most distinct from the other two?
# - bill_length_mm:
# - bill_depth_mm:
# - flipper_length_mm:
# - body_mass_g:

# conclusion for GQ3
# - what does this evidence suggest about morphological differences across species?
# A:

In [ ]:
# ------------------------------------------------------------
# inspect species across collection years
# ------------------------------------------------------------

# compare species counts and proportions within each collection year
#
# hint:
# OVER defines a window of rows over which an aggregate can be evaluated
# without collapsing those rows into a single result row
#
# PARTITION BY divides that window into independent groups
# here, the required denominator is:
# - the total number of observations within the corresponding year

db.sql("""
SELECT
    species,
    year,
    COUNT(*) AS n_observations,

    -- TODO: calculate the percentage of each species within its collection year
    -- hint: use SUM(COUNT(*)) OVER (...) to obtain the yearly total
    ROUND( 100.0 * COUNT(*) /
           SUM(COUNT(*)) OVER (...), 2 ) AS percent_within_year

FROM penguins_source
GROUP BY species, year
ORDER BY year, species;
""").show()

# expected result:
# 9 rows: 3 species × 3 collection years
# percentages should sum to approximately 100% within each year

In [ ]:
# ------------------------------------------------------------
# TODO
# interpret evidence for GQ4 — collection year
# ------------------------------------------------------------

# task:
# - use the species counts and within-year percentages obtained above
# - compare the species composition across collection years
# - conclude on GQ4

# E1. which species shows the largest variation in percentage across years?
# A1:

# E2. are the species proportions approximately stable across the three years?
# A2:

# conclusion for GQ4
# - what does this evidence suggest about year as a possible analytical attribute?
# A:

In [ ]:
# ------------------------------------------------------------
# guiding questions — attribute roles and provenance
# ------------------------------------------------------------

# IMPORTANT:
# these questions are not answered immediately
# they guide the following cells that collect evidence about source and processing attributes
# conclusions should be drawn only after inspecting that evidence

# prediction-time:
# assume that species is predicted,
# at the end of the field observation and measurement stage,
# i.e., prediction takes place before subsequent sample processing or laboratory assignment

# GQ5. what does sample_number reveal about the data-collection process?
# GQ6. how is batch assignment associated with species?
# GQ7. how is laboratory assignment associated with species, and how does it compare with batch?

In [ ]:
# ------------------------------------------------------------
# inspect sample_number across species and studies
# ------------------------------------------------------------

# compare sample_number ranges across study and species

db.sql("""
SELECT
    species,
    study,
    COUNT(*) AS n_observations,

    -- TODO: obtain the minimum sample_number in each species × study group
    ... AS min_sample_number,

    -- TODO: obtain the maximum sample_number in each species × study group
    ... AS max_sample_number

FROM penguins_source

-- TODO: define the groups needed to compare sample_number ranges
GROUP BY ...

ORDER BY species, study;
""").show()

# expected result:
# 9 rows: 3 species × 3 studies
#
# as a consistency check, across the three studies:
# - Adelie:    min = 1, max = 152
# - Chinstrap: min = 1, max = 68
# - Gentoo:    min = 1, max = 124
#
# inspect how the min/max ranges progress from one study to the next
# within each species

In [ ]:
# ------------------------------------------------------------
# inspect batch assignment across species
# ------------------------------------------------------------

# compare the distribution of batches across species

db.sql("""
SELECT
    species,
    name_batch,

    -- TODO: count the observations for each species × batch combination
    ... AS n_observations

FROM penguins_source

-- TODO: define the groups needed to compare batch assignment across species
GROUP BY ...

ORDER BY species, name_batch;
""").show()

# expected result:
# 12 rows
# each row represents one observed species × batch combination
# including combinations where name_batch is NULL

In [ ]:
# ------------------------------------------------------------
# inspect laboratory assignment across species
# ------------------------------------------------------------

# compare the distribution of laboratories across species

# note:
# recall that lab is not a penguin characteristic or an observation measurement
# it reflects subsequent data-processing / organizational context

db.sql("""
SELECT
    species,

    -- TODO: include the laboratory attribute
    ...,

    -- TODO: count the observations for each species × laboratory combination
    ... AS n_observations

FROM penguins_source

-- TODO: group observations by species and laboratory
GROUP BY ...

ORDER BY species, lab;
""").show()

# expected result:
# 9 rows
# each row represents one observed species × laboratory combination,
# including combinations where lab is NULL

In [ ]:
# ------------------------------------------------------------
# TODO
# interpret evidence for GQ5–GQ7 — attribute roles and provenance
# ------------------------------------------------------------

# task:
# - interpret the patterns shown by the previous results
# - use those patterns to conclude on GQ5, GQ6 and GQ7

# E1. how do sample_number ranges progress across studies within each species?
# A1:

# conclusion for GQ5
# A:

# E2. which species dominates each non-null batch, and what exceptions occur?
# A2:

# conclusion for GQ6
# A:

# E3. are all species represented in both laboratories, and how does this compare with the batch pattern?
# A3:

# conclusion for GQ7
# A:

In [ ]:
# ------------------------------------------------------------
# TODO
# select candidate attributes for the analytical dataset
# ------------------------------------------------------------

# task:
# - classify every attribute in penguins_source as:
#   target, retained candidate explanatory attribute, or excluded attribute
# - give one short justification for every retained or excluded attribute
#
# - base each decision on:
#   evidence collected above,
#   attribute meaning and provenance,
#   and prediction-time availability

# IMPORTANT:
# association with the target is not, by itself, sufficient reason
# to retain an attribute
#
# source or processing attributes may encode information about the target
# without representing legitimate prediction-time information

# target
# - ...

# retained candidate explanatory attributes
# - attribute:
#   justification:
# ===
# - attribute:
#   justification:
# ===
# ...

# excluded attributes
# - attribute:
#   justification:
# ===
# - attribute:
#   justification:
# ===
# ...

# final candidate explanatory attribute set
# - ...